## Text Generation Model

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping

## Data Gathering

In [2]:
with open("output.txt", "r", encoding="utf-8") as file:
    text = file.read()

print(text[:700])

I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.
"Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small sized unsalted. Not sure if this was an error or if the vendor intended to represent the product as ""Jumbo""."
"This is a confection that has been around a few centuries.  It is a light, pillowy citrus gelatin with nuts - in this case Filberts. And it is cut into tiny squares and then liberally coated with powdered sugar.  And it is a tiny mouthful o


## Tokenization

In [3]:
text=text[:70000]
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words=len(tokenizer.word_index)+1
print("Vocabulary Size:",total_words)

# Reverse mapping : index -> word
index_word = {i : word for word,i in tokenizer.word_index.items()}


Vocabulary Size: 2336


In [4]:
import joblib
joblib.dump(tokenizer, "tokenizer.pkl")

['tokenizer.pkl']

In [5]:
tokenizer.word_index

{'the': 1,
 'i': 2,
 'and': 3,
 'a': 4,
 'to': 5,
 'it': 6,
 'of': 7,
 'is': 8,
 'this': 9,
 'for': 10,
 'in': 11,
 'my': 12,
 'br': 13,
 'with': 14,
 'but': 15,
 'have': 16,
 'you': 17,
 'was': 18,
 'that': 19,
 'food': 20,
 'are': 21,
 'not': 22,
 'on': 23,
 'as': 24,
 'so': 25,
 'good': 26,
 'they': 27,
 'these': 28,
 'like': 29,
 'at': 30,
 'great': 31,
 'all': 32,
 'one': 33,
 'product': 34,
 'them': 35,
 'be': 36,
 'can': 37,
 'or': 38,
 "it's": 39,
 'we': 40,
 'taste': 41,
 'has': 42,
 'if': 43,
 'love': 44,
 'other': 45,
 'than': 46,
 'just': 47,
 'very': 48,
 'me': 49,
 'were': 50,
 'really': 51,
 'too': 52,
 'had': 53,
 'dog': 54,
 'flavor': 55,
 'up': 56,
 'when': 57,
 'will': 58,
 'more': 59,
 'from': 60,
 'coffee': 61,
 'better': 62,
 'oatmeal': 63,
 'much': 64,
 'use': 65,
 'little': 66,
 'sugar': 67,
 'your': 68,
 'only': 69,
 'about': 70,
 "i've": 71,
 'now': 72,
 'our': 73,
 'no': 74,
 'price': 75,
 "i'm": 76,
 "don't": 77,
 'because': 78,
 'time': 79,
 'get': 80,
 'am

In [6]:
index_word

{1: 'the',
 2: 'i',
 3: 'and',
 4: 'a',
 5: 'to',
 6: 'it',
 7: 'of',
 8: 'is',
 9: 'this',
 10: 'for',
 11: 'in',
 12: 'my',
 13: 'br',
 14: 'with',
 15: 'but',
 16: 'have',
 17: 'you',
 18: 'was',
 19: 'that',
 20: 'food',
 21: 'are',
 22: 'not',
 23: 'on',
 24: 'as',
 25: 'so',
 26: 'good',
 27: 'they',
 28: 'these',
 29: 'like',
 30: 'at',
 31: 'great',
 32: 'all',
 33: 'one',
 34: 'product',
 35: 'them',
 36: 'be',
 37: 'can',
 38: 'or',
 39: "it's",
 40: 'we',
 41: 'taste',
 42: 'has',
 43: 'if',
 44: 'love',
 45: 'other',
 46: 'than',
 47: 'just',
 48: 'very',
 49: 'me',
 50: 'were',
 51: 'really',
 52: 'too',
 53: 'had',
 54: 'dog',
 55: 'flavor',
 56: 'up',
 57: 'when',
 58: 'will',
 59: 'more',
 60: 'from',
 61: 'coffee',
 62: 'better',
 63: 'oatmeal',
 64: 'much',
 65: 'use',
 66: 'little',
 67: 'sugar',
 68: 'your',
 69: 'only',
 70: 'about',
 71: "i've",
 72: 'now',
 73: 'our',
 74: 'no',
 75: 'price',
 76: "i'm",
 77: "don't",
 78: 'because',
 79: 'time',
 80: 'get',
 81:

## Create input sequences: Instead of splitting line by line, use continuous text for better learning

In [7]:
text[:20]

'I have bought severa'

In [8]:
token_list = tokenizer.texts_to_sequences([text])[0]
input_sequences=[]

# Create sequences of 6 words
# First 7 words = input , 8th word =target
for i in range(7,len(token_list)):
  n_gram_sequence = token_list[i-7:i+1]
  input_sequences.append(n_gram_sequence)

print(input_sequences[:7])

[[2, 16, 113, 223, 7, 1, 1092, 530], [16, 113, 223, 7, 1, 1092, 530, 54], [113, 223, 7, 1, 1092, 530, 54, 20], [223, 7, 1, 1092, 530, 54, 20, 224], [7, 1, 1092, 530, 54, 20, 224, 3], [1, 1092, 530, 54, 20, 224, 3, 16], [1092, 530, 54, 20, 224, 3, 16, 100]]


In [9]:
token_list

[2,
 16,
 113,
 223,
 7,
 1,
 1092,
 530,
 54,
 20,
 224,
 3,
 16,
 100,
 35,
 32,
 5,
 36,
 7,
 26,
 103,
 1,
 34,
 718,
 59,
 29,
 4,
 1093,
 46,
 4,
 719,
 1094,
 3,
 6,
 531,
 62,
 12,
 1095,
 8,
 532,
 3,
 104,
 1096,
 9,
 34,
 62,
 46,
 201,
 34,
 225,
 1097,
 24,
 720,
 1098,
 275,
 1,
 275,
 50,
 202,
 203,
 721,
 1099,
 22,
 160,
 43,
 9,
 18,
 108,
 1100,
 38,
 43,
 1,
 533,
 1101,
 5,
 1102,
 1,
 34,
 24,
 720,
 9,
 8,
 4,
 1103,
 19,
 42,
 89,
 246,
 4,
 276,
 1104,
 6,
 8,
 4,
 1105,
 1106,
 722,
 1107,
 14,
 723,
 11,
 9,
 226,
 1108,
 3,
 6,
 8,
 322,
 247,
 323,
 1109,
 3,
 148,
 1110,
 1111,
 14,
 724,
 67,
 3,
 6,
 8,
 4,
 323,
 1112,
 7,
 1113,
 22,
 52,
 366,
 3,
 48,
 1114,
 2,
 277,
 140,
 9,
 324,
 278,
 43,
 17,
 21,
 534,
 14,
 1,
 1115,
 7,
 1116,
 279,
 1117,
 1,
 1118,
 1,
 725,
 3,
 1,
 1119,
 9,
 8,
 1,
 278,
 19,
 1120,
 1121,
 247,
 535,
 95,
 178,
 726,
 3,
 727,
 5,
 1,
 725,
 43,
 17,
 21,
 161,
 10,
 1,
 728,
 536,
 11,
 1122,
 2,
 280,
 2,
 16,
 100

## Padding
## Split X and Y

In [10]:
max_sequence_length =10
input_sequences = np.array(input_sequences)
X = input_sequences[:, :-1] # all sequences except last word/index
y = input_sequences[:, -1] # only last word/index


# Convert target to one-hot encoding
y = to_categorical(y, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (13026, 7)
y shape: (13026, 2336)


## Build LSTM Model

In [11]:
model = Sequential()
model.add(Input(shape=(max_sequence_length-1,)))
model.add(Embedding(input_dim=total_words,output_dim=128))
model.add(LSTM(150,return_sequences=True,dropout=0.2))
model.add(LSTM(100,dropout=0.2))
model.add(Dense(100,activation='relu'))
model.add(Dense(100,activation='relu'))
model.add(Dense(total_words,activation='softmax'))
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 9, 128)         │       299,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 9, 150)         │       167,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2336)           │       235,936 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 822,944 (3.14 MB)

 Trainable params: 822,944 (3.14 MB)

 Non-trainable params: 0 (0.00 B)

## Model Training

In [12]:
early_stop = EarlyStopping(monitor='loss',patience=5,restore_best_weights=True)

history = model.fit(X,y,epochs=50,batch_size=64,verbose=1,callbacks=[early_stop])

Epoch 1/50
204/204 ━━━━━━━━━━━━━━━━━━━━ 12s 32ms/step - accuracy: 0.0395 - loss: 6.6681
Epoch 2/50
204/204 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.0435 - loss: 6.2908
Epoch 3/50
204/204 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.0435 - loss: 6.1848
Epoch 4/50
204/204 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - accuracy: 0.0434 - loss: 6.0945
Epoch 5/50
204/204 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.0461 - loss: 5.9640
Epoch 6/50
204/204 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.0501 - loss: 5.8131
Epoch 7/50
204/204 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - accuracy: 0.0542 - loss: 5.6663
Epoch 8/50
204/204 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - accuracy: 0.0562 - loss: 5.5394
Epoch 9/50
204/204 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - accuracy: 0.0590 - loss: 5.4394
Epoch 10/50
204/204 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.0663 - loss: 5.3365
Epoch 11/50
204/204 ━━━━━━━━━━━━━━━━━━━━ 9s 44ms/step - accuracy: 0.0689 - loss: 5.2339
Epoch 12/50
204/204 ━━━━━━━━━━━━━━━━━━━━

## Using the trained model for final predictions
## TEMPERATURE + TOP-K SAMPLING
Temperature effect
0.5 → safer predictions
0.8 → balanced
1.2 → more creative/random

In [13]:
model.save("TextGenerationModel.keras")

In [14]:
def sample_with_temperature(preds, temperature=0.6, top_k=10):
    preds = np.asarray(preds).astype("float64")
    # [0.87,0.09,0.56,0.44,0.37,.............,0.89,0.32,...]

    # Select top k probabilities
    top_indices = np.argsort(preds)[-top_k:]
    # argsort : [0.09,0.32,0.37,0.44,0.56,0.87,0.89,........]
    # index of top k proabilities : [index]
    top_probs = preds[top_indices]
    #top k probs

    # Apply temperature scaling
    top_probs = np.log(top_probs + 1e-10) / temperature
    exp_probs = np.exp(top_probs)
    top_probs = exp_probs / np.sum(exp_probs)

    return np.random.choice(top_indices, p=top_probs)

## Text Generation method

In [15]:
def generate_text(seed_text, next_words=20):
    output_text = seed_text
    generated_words = []

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_length - 1,
            padding='pre'
        )

        predicted_probs = model.predict(token_list, verbose=0)[0]

        predicted_index = sample_with_temperature(
            predicted_probs,
            temperature=0.6,
            top_k=7  # ← increased from 7
        )

        next_word = index_word.get(predicted_index, "")

        # ✅ Only block IMMEDIATE repetition, not all past words
        if next_word and generated_words and next_word == generated_words[-1]:
            continue

        generated_words.append(next_word)
        output_text += " " + next_word

    return output_text

## Predictions

In [ ]:
print(generate_text("", next_words=15))

tasty food on the sea we can't believe that vendor amazon off too products br is were
